# Food Price Watch 04a: forecast choice on 2024
Compares "tomorrow = today" (naive), ETS and 16 LightGBM configurations (2 feature sets x 8 settings) on the **2024 validation year only**, at horizons 1, 7, 14 and 30 report days.
Picks the LightGBM configuration with the lowest relative error against naive (a simpler one within 1 % wins). No 2025-26 target is used here.
The choice is copied into `config/settings.yaml` (`forecast_choice`) and committed **before** notebook 04b scores 2025-26.
Writes `reports/tables/M1_validation_grid.csv` and `M2_validation_models.csv` (errors and ratios only). Run top to bottom and paste the last cell's output back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Code comes fresh from GitHub main each session; Drive holds data and outputs only.
import os
if not os.path.isdir('/content/foodprice_watch/.git'):
    !git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git /content/foodprice_watch
else:
    !git -C /content/foodprice_watch pull --ff-only origin main
%pip install -q -r /content/foodprice_watch/requirements-colab.txt

In [ ]:
import sys, os, json
os.chdir('/content/foodprice_watch')
sys.path.insert(0, '/content/foodprice_watch/src')
CODE_COMMIT = !git rev-parse --short HEAD
CODE_COMMIT = CODE_COMMIT[0]
from foodwatch.paths import SETTINGS, DATA_ROOT, RAW, data, raw, sha256, write_csv
# stop if Drive is not really mounted (a failed mount leaves an empty /content/drive on the Colab machine)
assert os.path.ismount('/content/drive'), 'Google Drive is not mounted: Runtime > Disconnect and delete runtime, then rerun cell 1'
assert DATA_ROOT.is_dir(), f'{DATA_ROOT} not found in Google Drive (wrong account, or the folder is missing)'
for folder in ['raw', 'processed', 'reports/tables']:
    (DATA_ROOT / folder).mkdir(parents=True, exist_ok=True)
for tmp in DATA_ROOT.rglob('*.tmp'):   # half-written files from an interrupted session
    tmp.unlink()
print('code commit:', CODE_COMMIT, '| data root:', DATA_ROOT)

In [ ]:
import pandas as pd
from foodwatch import forecast as fc, metrics as mt
D = pd.read_csv(data('data'), parse_dates=['Date'])
h_data = sha256(data('data'))
f = SETTINGS['forecast']; H = f['horizons']
S = fc.prepare(D, H, f['lags'], f['rolling'], f['rolling_min_periods'])
L = fc.stack(S, H)
dates = D.drop_duplicates('report_idx').set_index('report_idx').Date
val_start, val_end = SETTINGS['periods']['validation']
origins = fc.period_origins(dates, val_start, val_end, f['origin_step'], max(H))
print('validation origins:', len(origins), '|', dates[origins[0]].date(), 'to', dates[origins[-1]].date())

In [ ]:
# naive, ETS and the 16 LightGBM configurations on the 2024 origins
runs = [fc.rolling_forecasts(L, origins, fc.feature_columns('price_only', f['lags'], f['rolling']), fc.fit_naive, 10**9, 'naive'),
        fc.ets_forecasts(S, L, origins, H)]
configs = fc.grid(f['grid'])
for i, cfg in enumerate(configs):
    cols = fc.feature_columns(cfg['feature_set'], f['lags'], f['rolling'])
    runs.append(fc.rolling_forecasts(L, origins, cols, fc.make_fit_lgbm(cfg['params'], f['rounds']), f['refit_every'], cfg['config']))
    print(f'{i + 1}/{len(configs)}', cfg['config'])
V = pd.concat(runs, ignore_index=True)

In [ ]:
# Checks: only 2024 targets are scored; no training target after its fitting origin; nothing touches the 2022 gap; naive = 1.
Vs = mt.scored(V)
chk = dict(
    forecasts=len(V), scored=len(Vs),
    last_target_date=str(dates[int(Vs.target_idx.max())].date()),
    target_after_validation_year=int((Vs.target_idx.map(dates) > pd.Timestamp(val_end)).sum()),
    fit_after_origin=int((V.fit_cutoff > V.cutoff).sum()),
    rows_touching_gap=int(L.loc[L.report_idx.isin(origins), ['touch_gap', 'tgap']].any(axis=1).sum()),
    naive_rel_mae=round(float(mt.gmean_rel(V[V.model == 'naive']).gmean_rel_mae.max()), 6),
)
chk['ok'] = (chk['target_after_validation_year'] == 0 and chk['fit_after_origin'] == 0 and chk['rows_touching_gap'] == 0
             and chk['naive_rel_mae'] == 1.0)
assert chk['ok'], chk

In [ ]:
# Choice: geometric mean over items and horizons of the relative MAE against naive; a simpler configuration within the margin wins.
R = mt.rel_mae_by_item(V)
overall = R.groupby('model').rel.apply(mt.gmean)
order = [c['config'] for c in configs]
chosen = fc.choose(overall[order].to_dict(), order, f['tie_margin'])
cfg = next(c for c in configs if c['config'] == chosen)
G = mt.gmean_rel(V)
M1 = (G[G.model.isin(order)].pivot(index='model', columns='horizon', values='gmean_rel_mae').add_prefix('rel_h').reset_index()
      .assign(rel_all=lambda t: t.model.map(overall)).rename(columns={'model': 'config'}))
M1 = M1.set_index('config').loc[order].reset_index().round(4)
M1['chosen'] = M1.config == chosen
keep = V[V.model.isin(['naive', 'ets', chosen])].replace({'model': {chosen: 'lgbm'}})
M2 = mt.summary(keep).merge(mt.gmean_rel(keep), on=['model', 'horizon']).merge(mt.dm_vs_naive(keep), on=['model', 'horizon'], how='left')
M2 = M2.round(5)
T = DATA_ROOT / SETTINGS['files']['tables']
write_csv(M1, T / 'M1_validation_grid.csv')
write_csv(M2, T / 'M2_validation_models.csv')

In [ ]:
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
print('=== 04a forecast choice on 2024 (validation year only) ===')
print('code commit:', CODE_COMMIT, '| data_8.csv sha256', h_data[:16])
print('checks:', 'PASS' if chk['ok'] else 'FAIL', {k: v for k, v in chk.items() if k != 'ok'})
print('origins:', len(origins), '| horizons', H, '| rel = MAE of the model / MAE of "tomorrow = today" (below 1 = better than naive)')
print('\nM1 LightGBM grid, geometric mean relative MAE over items (simplest first)')
print(M1.to_string(index=False))
print('\nM2 naive, ETS and the chosen LightGBM by horizon (2024); dm_stat > 0 = worse than naive, < 0 = better; p_holm = Holm-corrected p; NaN = not computable')
print(M2[['model', 'horizon', 'n', 'mae_log', 'mae_log_naive', 'gmean_rel_mae', 'mae_tk', 'mae_tk_naive', 'accuracy', 'accuracy_naive',
          'direction_acc', 'dm_stat', 'p_holm']].to_string(index=False))
for m in ['ets', 'lgbm']:
    beats = [h for h in H if M2.loc[(M2.model == m) & (M2.horizon == h), 'gmean_rel_mae'].iloc[0] < 1]
    print(f'{m}: below naive at horizons {beats if beats else "none"}')
print('\nCHOICE (to be committed before 04b):')
print('forecast_choice:')
print(f"  config: {chosen}")
print(f"  feature_set: {cfg['feature_set']}")
print(f"  params: {cfg['params']}")
print(f"  validation_rel_mae: {round(float(overall[chosen]), 4)}")
print(f"  chosen_by: notebook 04a, code {CODE_COMMIT}, 2024 origins only")